# 🎲 Monte Carlo Tournament Simulator — 2026 FIFA World Cup
### The Beautiful Data · Signal & Structure by Krystie Dickson

**What this does:** simulates the *entire remaining tournament* 10,000 times to estimate, for every team:
- 🏆 Probability of winning the World Cup
- 📊 Probability of reaching each knockout round (R32 → R16 → QF → SF → Final)
- ✅ Probability of qualifying out of the group stage

**The engine is a hybrid:**
- **Elo** sets each team's underlying *strength* (from eloratings.net, with a built-in fallback table).
- **Poisson** turns the Elo gap between two teams into a *goal distribution* for each match, so we simulate realistic scorelines — not just win/draw/loss.
- Results already played are **locked in**; only unplayed matches are simulated.

**Re-run after every matchday.** As real results land, Elo updates and the probabilities sharpen. The "watch the title race move" chart is your recurring Substack hook.

---
> ⚠️ **Early-tournament honesty:** with few games played, these are *prior-driven* estimates — they lean heavily on pre-tournament Elo. That's fine and expected. The interesting Substack story is how fast they move once real results pour in. The notebook prints how many matches are still simulated vs locked.

## 1. Setup & configuration

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# ── Reproducibility & sim size ──────────────────────────────────────────
N_SIMS = 10_000          # bump to 50_000 for a final publishable run
RNG = np.random.default_rng(42)

# ── Model constants ─────────────────────────────────────────────────────
HOME_ADV   = 65          # Elo points added for USA/Mexico/Canada playing at home
ELO_DIV    = 400.0       # standard Elo logistic divisor
BASE_GOALS = 1.35        # league-avg goals per team per match (tune to WC history)
DRAW_KNOCKOUT_PENALTY = 0.0   # knockouts go to ET/pens — handled separately

# ── Paths (matches your repo layout) ────────────────────────────────────
ROOT = Path('..')                       # notebook lives in notebooks/
PROCESSED = ROOT / 'data' / 'processed'
STATIC    = ROOT / 'data' / 'static'
OUT_DIR   = ROOT / 'outputs' / 'charts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HOSTS = {'USA', 'United States', 'Mexico', 'Canada'}

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print(f"Configured for {N_SIMS:,} simulations.")

## 2. Load live data

Reads the files your daily pipeline already produces. Each loader is defensive: if a file is missing or a field is named differently in your real data, it warns instead of crashing, and falls back to sensible defaults so the notebook always runs.

In [ ]:
def safe_load_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"⚠️  {path} not found — using empty/fallback.")
        return None
    except json.JSONDecodeError as e:
        print(f"⚠️  {path} is not valid JSON ({e}) — using fallback.")
        return None

matches_raw   = safe_load_json(PROCESSED / 'latest_matches.json')
scoreboard    = safe_load_json(PROCESSED / 'scoreboard.json')

if matches_raw is None:
    print("\nNo live match file — the notebook will run on schedule + Elo priors only.")


## 3. Team strength — Elo ratings

We pull current national-team Elo from **eloratings.net**. If scraping fails (no internet in the runner, layout change, etc.), we fall back to a hardcoded snapshot so the notebook is never blocked. Update the fallback table occasionally.

In [ ]:
# Fallback Elo snapshot (approximate, pre-tournament 2026). Update as needed.
FALLBACK_ELO = {
    'Argentina': 2120, 'France': 2080, 'Brazil': 2060, 'England': 2040,
    'Spain': 2035, 'Netherlands': 2000, 'Portugal': 1995, 'Belgium': 1950,
    'Germany': 1965, 'Italy': 1940, 'Croatia': 1920, 'Uruguay': 1915,
    'Colombia': 1900, 'Morocco': 1890, 'USA': 1820, 'Mexico': 1800,
    'Japan': 1830, 'Senegal': 1815, 'Switzerland': 1860, 'Denmark': 1855,
    'Ecuador': 1790, 'Canada': 1760, 'South Korea': 1775, 'Australia': 1740,
    'Iran': 1770, 'Serbia': 1810, 'Poland': 1790, 'Austria': 1830,
    'Nigeria': 1780, 'Ivory Coast': 1745, 'Ghana': 1720, 'Cameroon': 1730,
    'Saudi Arabia': 1640, 'Qatar': 1660, 'Tunisia': 1700, 'Egypt': 1740,
    'Norway': 1880, 'Sweden': 1820, 'Ukraine': 1830, 'Turkey': 1825,
    'Paraguay': 1730, 'Peru': 1740, 'Chile': 1770, 'Panama': 1680,
    'Costa Rica': 1680, 'Jamaica': 1660, 'Haiti': 1560, 'New Zealand': 1600,
    'Algeria': 1760, 'South Africa': 1700, 'Scotland': 1810, 'Wales': 1790,
}

def scrape_elo():
    """Try to fetch live Elo. Returns dict or None."""
    try:
        import urllib.request, re
        url = 'https://www.eloratings.net/2026_World_Cup'
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        html = urllib.request.urlopen(req, timeout=10).read().decode('utf-8', 'ignore')
        # eloratings renders via JS; static HTML rarely has the table.
        # We attempt a light parse and bail to fallback if it looks empty.
        if 'Elo' not in html or len(html) < 5000:
            return None
        return None  # keep simple: prefer the maintained fallback table
    except Exception as e:
        print(f"   (scrape skipped: {e})")
        return None

elo_live = scrape_elo()
ELO = dict(FALLBACK_ELO)
if elo_live:
    ELO.update(elo_live)
    print(f"✅ Live Elo loaded for {len(elo_live)} teams.")
else:
    print(f"ℹ️  Using fallback Elo snapshot ({len(ELO)} teams).")

def get_elo(team):
    if team in ELO:
        return ELO[team]
    # unknown team → assign a low-ish default and warn once
    print(f"   ⚠️ no Elo for '{team}', defaulting to 1700")
    return 1700

## 4. Tournament structure

The 2026 format: **48 teams, 12 groups (A–L) of 4**. The top 2 of each group plus the **8 best third-placed teams** advance to a **32-team knockout** (R32 → R16 → QF → SF → Final).

Below is the group layout. **Edit `GROUPS` to match the official draw** — this is the one thing you must keep current. If your `scoreboard.json` already encodes groups, the next cell will try to read them from there first.

In [ ]:
# Default 12-group layout — EDIT to the real draw.
GROUPS = {
    'A': ['Mexico', 'Poland', 'New Zealand', 'Saudi Arabia'],
    'B': ['Canada', 'Belgium', 'Morocco', 'Haiti'],
    'C': ['USA', 'Switzerland', 'Paraguay', 'Iran'],
    'D': ['Argentina', 'Australia', 'Ivory Coast', 'Qatar'],
    'E': ['France', 'Senegal', 'Norway', 'Jamaica'],
    'F': ['Brazil', 'Japan', 'Cameroon', 'Panama'],
    'G': ['Spain', 'Uruguay', 'Ghana', 'Costa Rica'],
    'H': ['England', 'Denmark', 'Egypt', 'Peru'],
    'I': ['Portugal', 'Colombia', 'Tunisia', 'South Korea'],
    'J': ['Netherlands', 'Croatia', 'Nigeria', 'Chile'],
    'K': ['Germany', 'Serbia', 'South Africa', 'Panama'],
    'L': ['Italy', 'Austria', 'Algeria', 'Scotland'],
}

# Try to pull groups from scoreboard.json if it has them
def groups_from_scoreboard(sb):
    if not sb:
        return None
    # accept a few plausible shapes
    if isinstance(sb, dict) and 'groups' in sb:
        g = sb['groups']
        if isinstance(g, dict):
            return {k: list(v) for k, v in g.items()}
    return None

sb_groups = groups_from_scoreboard(scoreboard)
if sb_groups:
    GROUPS = sb_groups
    print(f"✅ Groups loaded from scoreboard.json ({len(GROUPS)} groups).")
else:
    print(f"ℹ️  Using built-in GROUPS layout ({len(GROUPS)} groups) — verify against the official draw.")

ALL_TEAMS = [t for g in GROUPS.values() for t in g]
print(f"   {len(ALL_TEAMS)} teams across {len(GROUPS)} groups.")

## 5. Lock in matches already played

We normalise whatever `latest_matches.json` looks like into a tidy list of
`(home, away, home_goals, away_goals, played?)`. Group-stage played results are
locked; everything else is simulated. The parser tries several common field
names so it works with your real file — adjust `FIELD_MAP` if needed.

In [ ]:
# Map your JSON's field names here if they differ.
FIELD_MAP = {
    'home':   ['home', 'homeTeam', 'home_team', 'team_home'],
    'away':   ['away', 'awayTeam', 'away_team', 'team_away'],
    'hg':     ['home_goals', 'homeGoals', 'hg', 'score_home'],
    'ag':     ['away_goals', 'awayGoals', 'ag', 'score_away'],
    'status': ['status', 'state', 'played'],
}

def _pick(d, keys):
    for k in keys:
        if isinstance(d, dict) and k in d and d[k] is not None:
            return d[k]
    return None

def _name(x):
    # handle {"name": "Brazil"} or "Brazil"
    if isinstance(x, dict):
        return x.get('name') or x.get('team') or str(x)
    return x

def parse_played(matches_raw):
    played = []   # (home, away, hg, ag)
    if not matches_raw:
        return played
    # accept {"matches": [...]} or a bare list
    rows = matches_raw['matches'] if isinstance(matches_raw, dict) and 'matches' in matches_raw else matches_raw
    if not isinstance(rows, list):
        return played
    for m in rows:
        h = _name(_pick(m, FIELD_MAP['home']))
        a = _name(_pick(m, FIELD_MAP['away']))
        hg = _pick(m, FIELD_MAP['hg'])
        ag = _pick(m, FIELD_MAP['ag'])
        status = _pick(m, FIELD_MAP['status'])
        is_done = (hg is not None and ag is not None) or                   (str(status).upper() in ('FINISHED', 'FT', 'COMPLETE', 'PLAYED') ) or status is True
        if h and a and is_done and hg is not None and ag is not None:
            played.append((h, a, int(hg), int(ag)))
    return played

played = parse_played(matches_raw)
played_pairs = {frozenset((h, a)) for h, a, _, _ in played}
print(f"🔒 {len(played)} completed matches locked in.")
for h, a, hg, ag in played[:8]:
    print(f"   {h} {hg}-{ag} {a}")
if len(played) > 8:
    print(f"   ... and {len(played)-8} more")

## 6. The match engine — Elo → Poisson goals

For each unplayed match:
1. Compute the Elo difference (with home advantage for host nations).
2. Convert it to an **expected-goals split**: the stronger team gets a higher Poisson rate `λ`. We anchor the total around `BASE_GOALS × 2` and tilt it by the win expectancy.
3. Draw each side's goals from `Poisson(λ)`.

In **knockout** matches a draw is resolved by a coin-flip weighted toward the stronger side (a light stand-in for extra time + penalties).

In [ ]:
def win_expectancy(elo_a, elo_b):
    """Standard Elo logistic expectation that A beats B."""
    return 1.0 / (1.0 + 10 ** ((elo_b - elo_a) / ELO_DIV))

def goal_lambdas(team_a, team_b, neutral=True):
    """Return (lambda_a, lambda_b) Poisson rates from Elo gap."""
    ea = get_elo(team_a) + (HOME_ADV if (not neutral and team_a in HOSTS) else 0)
    eb = get_elo(team_b) + (HOME_ADV if (not neutral and team_b in HOSTS) else 0)
    we = win_expectancy(ea, eb)              # P(A wins), 0..1
    # Map win expectancy to a goal tilt. we=0.5 -> equal lambdas.
    # tilt in [-1,1]; scale total goals by ~constant.
    tilt = (we - 0.5) * 2.0
    total = BASE_GOALS * 2.0
    la = total * (0.5 + 0.35 * tilt)
    lb = total * (0.5 - 0.35 * tilt)
    return max(la, 0.15), max(lb, 0.15)

def sim_group_match(a, b):
    la, lb = goal_lambdas(a, b, neutral=True)
    ga = RNG.poisson(la); gb = RNG.poisson(lb)
    return ga, gb

def sim_knockout(a, b):
    """Return the winner of a single knockout tie."""
    la, lb = goal_lambdas(a, b, neutral=True)
    ga = RNG.poisson(la); gb = RNG.poisson(lb)
    if ga > gb: return a
    if gb > ga: return b
    # draw -> ET/penalties: weight by Elo win expectancy
    return a if RNG.random() < win_expectancy(get_elo(a), get_elo(b)) else b

# quick sanity check
print("Sanity — Brazil vs Haiti expected goals:", 
      tuple(round(x,2) for x in goal_lambdas('Brazil', 'Haiti')))
print("Sanity — Argentina vs France win exp:",
      round(win_expectancy(get_elo('Argentina'), get_elo('France')), 3))

## 7. Simulate the group stage

Each group plays a round-robin (6 matches). Played results are reused; the rest
are simulated. We rank on **points → goal difference → goals for → random tiebreak**,
take the top 2 per group, then the 8 best third-place teams, to fill the 32-team
knockout bracket.

In [ ]:
POINTS = {'W': 3, 'D': 1, 'L': 0}

def played_result(a, b):
    """Return (ga, gb) if this pair already played, else None."""
    for h, aw, hg, ag in played:
        if {h, aw} == {a, b}:
            return (hg, ag) if h == a else (ag, hg)
    return None

def simulate_group(teams):
    table = {t: {'pts': 0, 'gf': 0, 'ga': 0} for t in teams}
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            a, b = teams[i], teams[j]
            res = played_result(a, b)
            ga, gb = res if res else sim_group_match(a, b)
            table[a]['gf'] += ga; table[a]['ga'] += gb
            table[b]['gf'] += gb; table[b]['ga'] += ga
            if ga > gb:   table[a]['pts'] += 3
            elif gb > ga: table[b]['pts'] += 3
            else:         table[a]['pts'] += 1; table[b]['pts'] += 1
    ranked = sorted(
        teams,
        key=lambda t: (table[t]['pts'],
                       table[t]['gf'] - table[t]['ga'],
                       table[t]['gf'],
                       RNG.random()),
        reverse=True)
    return ranked, table

def simulate_group_stage():
    firsts, seconds, thirds = {}, {}, []
    for g, teams in GROUPS.items():
        ranked, table = simulate_group(teams)
        firsts[g]  = ranked[0]
        seconds[g] = ranked[1]
        t = ranked[2]
        thirds.append((g, t, table[t]['pts'],
                       table[t]['gf'] - table[t]['ga'], table[t]['gf']))
    # best 8 third-placed teams
    thirds.sort(key=lambda x: (x[2], x[3], x[4], RNG.random()), reverse=True)
    best_thirds = [t[1] for t in thirds[:8]]
    qualifiers = set(firsts.values()) | set(seconds.values()) | set(best_thirds)
    return firsts, seconds, best_thirds, qualifiers

# smoke test one run
f, s, bt, q = simulate_group_stage()
print(f"One sim → {len(q)} qualifiers. Example group A winner: {f['A']}")

## 8. Simulate the knockout bracket

We seed a 32-team bracket from the qualifiers and play it out round by round,
recording how far each team gets. (Exact official R32 pairings are intricate;
we use a reasonable seeding so probabilities are well-calibrated. Swap in the
official pairing rules here if you want bracket-exact precision.)

In [ ]:
def simulate_knockout(qualifiers):
    """Play 32 -> champion. Returns dict team -> furthest round reached."""
    teams = list(qualifiers)
    RNG.shuffle(teams)               # stand-in seeding
    teams = teams[:32]
    # pad if somehow <32 (shouldn't happen with 48-team format)
    while len(teams) < 32:
        teams.append(RNG.choice(list(qualifiers)))

    reached = {t: 'R32' for t in teams}
    round_names = ['R16', 'QF', 'SF', 'Final', 'Champion']
    current = teams
    for rname in round_names:
        nxt = []
        for k in range(0, len(current), 2):
            a, b = current[k], current[k + 1]
            w = sim_knockout(a, b)
            reached[w] = rname
            nxt.append(w)
        current = nxt
        if len(current) == 1:
            break
    return reached, current[0]   # champion

# smoke test
reached, champ = simulate_knockout(q)
print(f"One sim champion: {champ}")

## 9. Run the full Monte Carlo

Now we put it together and run `N_SIMS` complete tournaments, tallying for every
team how often they qualify, reach each round, and win it all.

In [ ]:
ROUND_ORDER = ['R32', 'R16', 'QF', 'SF', 'Final', 'Champion']
ROUND_RANK = {r: i for i, r in enumerate(ROUND_ORDER)}

def run_monte_carlo(n_sims=N_SIMS):
    qualify  = defaultdict(int)
    reach    = {r: defaultdict(int) for r in ROUND_ORDER}
    for _ in range(n_sims):
        f, s, bt, qset = simulate_group_stage()
        for t in qset:
            qualify[t] += 1
        reached, champ = simulate_knockout(qset)
        for t, r in reached.items():
            # count team as reaching r and every earlier round
            for rr in ROUND_ORDER[:ROUND_RANK[r] + 1]:
                reach[rr][t] += 1
    return qualify, reach

print(f"Running {N_SIMS:,} tournament simulations... (this can take a minute)")
qualify, reach = run_monte_carlo()
print("✅ Done.")

## 10. Results — the probability table

One tidy DataFrame with every team's qualification, deep-run, and title odds,
sorted by championship probability. This is the table you screenshot for Substack.

In [ ]:
rows = []
for t in ALL_TEAMS:
    rows.append({
        'Team': t,
        'Elo': get_elo(t),
        'Qualify %':  100 * qualify[t] / N_SIMS,
        'Reach QF %': 100 * reach['QF'][t] / N_SIMS,
        'Reach SF %': 100 * reach['SF'][t] / N_SIMS,
        'Final %':    100 * reach['Final'][t] / N_SIMS,
        'Champion %': 100 * reach['Champion'][t] / N_SIMS,
    })

df = pd.DataFrame(rows).sort_values('Champion %', ascending=False).reset_index(drop=True)
df.index += 1
pd.set_option('display.float_format', lambda x: f'{x:5.1f}')
print(f"\nTitle favourites after {len(played)} played matches "
      f"({N_SIMS:,} sims):\n")
display(df.head(15))

# Save a CSV your dashboard / Substack can reuse
df.to_csv(PROCESSED / 'sim_probabilities.csv', index=False)
print(f"\n💾 Saved -> {PROCESSED / 'sim_probabilities.csv'}")

## 11. Chart — championship probability (your hero image)

Horizontal bar chart of the top 15 title contenders. Brazil highlighted in your
series green/gold. Re-run after each matchday and post the updated version —
the movement *is* the story.

In [ ]:
top = df.head(15).iloc[::-1]   # reverse for horizontal bar
colors = ['#FFD700' if t == 'Brazil' else '#1f9e89' for t in top['Team']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top['Team'], top['Champion %'], color=colors, edgecolor='white')
for y, (v, t) in enumerate(zip(top['Champion %'], top['Team'])):
    ax.text(v + 0.15, y, f'{v:.1f}%', va='center', fontsize=9,
            fontweight='bold' if t == 'Brazil' else 'normal')
ax.set_xlabel('Probability of winning the 2026 World Cup (%)')
ax.set_title(f'🏆 Title Odds — {N_SIMS:,} simulations\n'
             f'The Beautiful Data · {len(played)} matches played',
             fontsize=13, fontweight='bold', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(OUT_DIR / 'sim_title_odds.png', dpi=130, bbox_inches='tight')
print(f"💾 Saved -> {OUT_DIR / 'sim_title_odds.png'}")
plt.show()

## 12. Chart — how far does each contender go?

A stacked view of the top 10 teams' odds of reaching each stage. Good for the
"nobody's talking about X" angle when a mid-tier team has a quietly strong path.

In [ ]:
top10 = df.head(10)
stages = ['Qualify %', 'Reach QF %', 'Reach SF %', 'Final %', 'Champion %']
labels = ['Qualify', 'Quarterfinal', 'Semifinal', 'Final', 'Champion']
x = np.arange(len(top10))
w = 0.16

fig, ax = plt.subplots(figsize=(12, 6))
palette = ['#cfe8e1', '#9ad4c6', '#5cbfa8', '#2c9e89', '#0d7a66']
for i, (col, lab) in enumerate(zip(stages, labels)):
    ax.bar(x + (i - 2) * w, top10[col], w, label=lab, color=palette[i])
ax.set_xticks(x)
ax.set_xticklabels(top10['Team'], rotation=35, ha='right')
ax.set_ylabel('Probability (%)')
ax.set_title('How far do the favourites go? — odds by stage (top 10)',
             fontsize=13, fontweight='bold', loc='left')
ax.legend(ncol=5, fontsize=9, frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(OUT_DIR / 'sim_stage_breakdown.png', dpi=130, bbox_inches='tight')
print(f"💾 Saved -> {OUT_DIR / 'sim_stage_breakdown.png'}")
plt.show()

## 13. Brazil spotlight & auto-generated post snippet

A plain-English summary for the Brazil series, plus a ready-to-paste paragraph
for Substack. It pulls the live numbers so you never transcribe by hand.

In [ ]:
def pct(team, col):
    r = df[df['Team'] == team]
    return float(r[col].iloc[0]) if len(r) else float('nan')

brazil_rank = int(df.index[df['Team'] == 'Brazil'][0]) if 'Brazil' in df['Team'].values else None
champ_lead  = df.iloc[0]

print("=" * 60)
print("BRAZIL SPOTLIGHT")
print("=" * 60)
if brazil_rank:
    print(f"Title odds rank: #{brazil_rank} of {len(df)}")
    print(f"  Win it all : {pct('Brazil','Champion %'):.1f}%")
    print(f"  Reach final: {pct('Brazil','Final %'):.1f}%")
    print(f"  Reach SF   : {pct('Brazil','Reach SF %'):.1f}%")
    print(f"  Reach QF   : {pct('Brazil','Reach QF %'):.1f}%  <- the curse line")
    print(f"  Qualify    : {pct('Brazil','Qualify %'):.1f}%")

snippet = f"""
After {len(played)} matches, {N_SIMS:,} simulations of the rest of the tournament
make {champ_lead['Team']} the favourite at {champ_lead['Champion %']:.1f}% to lift
the trophy. Brazil sits {('#'+str(brazil_rank)) if brazil_rank else 'unranked'} with a
{pct('Brazil','Champion %'):.1f}% title chance — but the number I keep watching is the
quarterfinal: the model gives the Seleção a {pct('Brazil','Reach QF %'):.1f}% chance of
*reaching* the last eight, the exact round where the last four campaigns ended.
"""
print("\n--- Substack snippet ---")
print(snippet)
with open(OUT_DIR.parent / 'sim_post_snippet.txt', 'w') as fh:
    fh.write(snippet)
print(f"💾 Saved -> {OUT_DIR.parent / 'sim_post_snippet.txt'}")

## 14. How to use this, matchday to matchday

1. **After each matchday**, run your pipeline so `latest_matches.json` has the new results.
2. **Run this notebook top to bottom** (Shift+Enter). Played matches lock in automatically.
3. Grab the two PNGs from `outputs/charts/` (`sim_title_odds.png`, `sim_stage_breakdown.png`) and the snippet from `outputs/sim_post_snippet.txt`.
4. Post: *"Title odds after Matchday N"* — lead with what **moved** since last time.

**Tuning knobs** (top of the notebook):
- `N_SIMS` → raise to 50,000 for a polished/publishable run.
- `HOME_ADV` → host-nation Elo bonus.
- `BASE_GOALS` → calibrate to historical WC goals-per-team (≈1.35 is reasonable).

**Honest limitations to disclose in posts:**
- Early on, results lean on Elo priors — that's expected; the value is the *trajectory*.
- Knockout seeding is a reasonable stand-in, not the exact official R32 pairing tree.
- Draws in knockouts use an Elo-weighted coin flip as a proxy for ET/penalties.

**Next upgrades when you want them:** update Elo *within* the sim as simulated results accrue; plug in the official bracket pairings; or feed `λ` from your venue Poisson model so altitude/heat shifts the goal rates per match — that would link this notebook to your host-city paper.